In [0]:
import json
from typing import Dict, Any, List
from pyspark.sql import SparkSession, DataFrame, Window
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType

In [0]:
%run ../Shared/CommonMethods/ABC/FileHandling

In [0]:
%run ../Shared/CommonMethods/ABC/ErrorHandling

In [0]:
%run ../Shared/CommonMethods/ABC/CreateUserDefinedFunctions

In [0]:
# ==============================================================================
# Setup Parameters (Widgets Setup)
# ==============================================================================
dbutils.widgets.text("FileID", "", "")
dbutils.widgets.text("CurrentContainer", "", "")
dbutils.widgets.text("CurrentFolderPath", "", "")
dbutils.widgets.text("FileName", "", "")
dbutils.widgets.text("Delimiter", "", "")
dbutils.widgets.text("HasHeaders", "", "")
dbutils.widgets.text("IgnoreHeader", "", "")
dbutils.widgets.text("TextQualifier", "", "")
dbutils.widgets.text("ValidatedFolderPath", "", "")
dbutils.widgets.text("ValidationContainer", "", "")
dbutils.widgets.text("ValidationFileFolderPath", "", "")
dbutils.widgets.text("ValidationFileName", "", "")

In [0]:
# ==============================================================================
# Global Configuration Variables Parsing
# ==============================================================================
mount_point = "/mnt/"
file_id = dbutils.widgets.get("FileID")
current_container = dbutils.widgets.get("CurrentContainer")
current_folder_path = dbutils.widgets.get("CurrentFolderPath")
file_name = dbutils.widgets.get("FileName")

validated_folder_path = dbutils.widgets.get("ValidatedFolderPath")
validation_container = dbutils.widgets.get("ValidationContainer")
validation_file_folder_path = dbutils.widgets.get("ValidationFileFolderPath")
validation_file_name = dbutils.widgets.get("ValidationFileName")

delimiter = dbutils.widgets.get("Delimiter") or "|"
header = dbutils.widgets.get("HasHeaders")
ignore_header = dbutils.widgets.get("IgnoreHeader")
text_qualifier = dbutils.widgets.get("TextQualifier")

In [0]:
# Parse fileName safely without extension
error_folder_name = file_name
for ext in [".txt", ".TXT", ".csv", ".CSV"]:
    if ext in file_name:
        error_folder_name = file_name.rsplit(".", 1)[0]
        break

In [0]:
full_file = f"{mount_point}{current_container}{current_folder_path}/{file_name}"
full_validation = f"{mount_point}{validation_container}{validation_file_folder_path}/{validation_file_name}"
full_error = f"{mount_point}{validated_folder_path}/Error/{error_folder_name}.err"

In [0]:
def output_validation_error(df: DataFrame, path: str) -> None:
    # Mimics outputValidationError from referenced legacy Scala ErrorHandling notebook
    df.write.mode("append").format("delta").save(path)

In [0]:
def is_ignore_header(data_path: str, schema_path: str, dlm: str, qte: str) -> DataFrame:
    # Mimics isIgnoreHeader template logic from FileHandling dependency
    return spark.read.option("header", "false").format("csv").option("delimiter", dlm).option("quote", qte).load(data_path)

In [0]:
def without_header(data_path: str, schema_path: str, dlm: str, qte: str) -> DataFrame:
    # Mimics withoutHeader template logic from FileHandling dependency
    return spark.read.option("header", "false").format("csv").option("delimiter", dlm).option("quote", qte).load(data_path)

In [0]:
class ValidationCoreEngine:
    """Encapsulates all processing metrics evaluations targeting dataframes rows."""

    @staticmethod
    def output_checks(df: DataFrame, error_path: str) -> int:
        count = df.count()
        if count > 0:
            output_validation_error(df, error_path)
        return count
    
    def is_required(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator)) \
            .withColumn("ERROR_DESCRIPTION", F.lit("Field Missing")) \
            .filter((F.col("VALUE").isNull()) | (F.col("VALUE").cast("string") == ""))
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)
    
    def is_required_with_condition(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        cond_col = kwargs.get("ConditionCol", "")
        cond_vals = kwargs.get("ConditionVal", [])
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE"), cond_col) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator)) \
            .withColumn("ERROR_DESCRIPTION", F.concat(F.lit("Field Missing when "), F.lit(cond_col), F.lit(" is "), F.lit(", ".join(cond_vals)))) \
            .where(F.col(cond_col).isin(cond_vals)) \
            .filter((F.col("VALUE").isNull()) | (F.trim(F.col("VALUE")) == ""))
            
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)

    def is_date_not_required(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        fmt = kwargs.get("Value", "yyyy-MM-dd")
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .withColumn("new_date", F.from_unixtime(F.unix_timestamp(F.col("VALUE").cast(StringType()), fmt), "yyyy-MM-dd")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator)) \
            .withColumn("ERROR_DESCRIPTION", F.lit("Not a valid date or format when populated")) \
            .where(F.trim(F.col("VALUE")) != "") \
            .filter(F.col("new_date").isNull())
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)
    
    def is_data_type(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        d_type = kwargs.get("Value", "string")
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator)) \
            .withColumn("ERROR_DESCRIPTION", F.lit("Incorrect Datatype")) \
            .filter(F.col("VALUE").cast(d_type).isNull())
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)

    def is_data_type_not_required(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        d_type = kwargs.get("Value", "string")
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator)) \
            .withColumn("ERROR_DESCRIPTION", F.lit("Incorrect Datatype")) \
            .where(F.trim(F.col("VALUE")) != "''") \
            .filter(F.col("VALUE").cast(d_type).isNull())
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)

    def is_in_allowed_values(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        allowed = kwargs.get("InputValue", [])
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator)) \
            .withColumn("ERROR_DESCRIPTION", F.lit("Not in Allowed Values")) \
            .filter(~F.col("VALUE").isin(allowed))
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)

    def is_min_value(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        min_val = int(kwargs.get("Value", 0))
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator)) \
            .withColumn("ERROR_DESCRIPTION", F.lit("Incorrect MinValue")) \
            .filter(F.col("VALUE").cast(IntegerType()) < min_val)
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)

    def is_max_value(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        max_val = int(kwargs.get("Value", 0))
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator)) \
            .withColumn("ERROR_DESCRIPTION", F.lit("Incorrect MaxValue")) \
            .filter(F.col("VALUE").cast(IntegerType()) > max_val)
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)

    def is_max_date(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        fmt = kwargs.get("Value", "yyyy-MM-dd")
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .filter((F.col("VALUE").isNotNull()) | (F.col("VALUE").cast("string") != "")) \
            .withColumn("new_date", F.from_unixtime(F.unix_timestamp(F.col("VALUE").cast(StringType()), fmt), "yyyy-MM-dd")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator)) \
            .withColumn("ERROR_DESCRIPTION", F.lit("Not a valid date, greater than current date")) \
            .filter(F.col("new_date") > F.current_date())
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)

    def data_length_check(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        length_val = int(kwargs.get("Value", 0))
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator)) \
            .withColumn("ERROR_DESCRIPTION", F.lit("Incorrect Data Length")) \
            .withColumn("VALUE_CLEAN", F.regexp_replace(F.col("VALUE"), "\\s+", "")) \
            .withColumn("ACTUAL_DATA_LENGTH", F.length("VALUE_CLEAN")) \
            .filter((F.col("ACTUAL_DATA_LENGTH") != length_val) & (F.col("ACTUAL_DATA_LENGTH") != 0))
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)

    def is_mon_ccyy_valid(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .filter((F.col("VALUE").isNotNull()) | (F.col("VALUE").cast("string") != "")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("MON", F.substring(F.col("VALUE"), 0, 3)) \
            .withColumn("YEAR", F.substring(F.col("VALUE"), -4, 4)) \
            .withColumn("VALIDATOR", F.lit("isMONCCYYValid")) \
            .withColumn("ERROR_DESCRIPTION", F.lit("Not a valid MONCCYY")) \
            .filter(
                (~F.col("YEAR").rlike("^[0-9]*$")) |
                ((F.col("YEAR").rlike("^[0-9]*$")) & (F.col("YEAR").cast(IntegerType()) < 1900)) |
                (~F.upper(F.col("MON")).rlike("JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC")) |
                (F.length(F.col("VALUE")) != 7)
            )
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)

    def is_in_allowed_values_and_null(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        allowed = kwargs.get("InputValue", [])
        df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .filter((F.col("VALUE").isNotNull()) & (F.col("VALUE").cast("string") != "")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator)) \
            .withColumn("ERROR_DESCRIPTION", F.lit("Not in Allowed Values")) \
            .filter(~F.col("VALUE").isin(allowed))
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)
    
    def is_over_punch(self, df_data: DataFrame, col_name: str, validator: str, error_path: str, **kwargs) -> int:
        temp_df = df_data.select("FILE_ID", "ROW_ID", F.col(col_name).alias("VALUE")) \
            .filter((F.col("VALUE").isNotNull()) & (F.col("VALUE").cast("string") != "")) \
            .withColumn("FIELD_NAME", F.lit(col_name)) \
            .withColumn("VALIDATOR", F.lit(validator))
            
        temp_df.createOrReplaceTempView("TestDF")
        # overpunch() is registered as a Hive/Spark UDF via 'CreateUserDefinedFunctions'
        validation_sql = """
            SELECT FILE_ID, ROW_ID, VALUE, FIELD_NAME, VALIDATOR,
                   'Is not an overpunch character' AS ERROR_DESCRIPTION,
                   overpunch(VALUE) AS IsOverpunch
            FROM TestDF
        """
        df = spark.sql(validation_sql).filter(F.col("IsOverpunch").isNull())
        return self.output_checks(df.select("FILE_ID", "ROW_ID", "VALUE", "FIELD_NAME", "VALIDATOR", "ERROR_DESCRIPTION"), error_path)


In [0]:
# ==============================================================================
# Orchestration Handler Routine
# ==============================================================================
def run_level2_validator(schema_path: str, data_path: str):
    spark_session = SparkSession.builder.getOrCreate()
    engine = ValidationCoreEngine()
    
    # Validation strategy routing registry maps exact legacy targets
    validator_registry = {
        "isRequired": engine.is_required,
        "isDate": engine.is_date,
        "isMaxDate": engine.is_max_date,
        "isDateNotRequired": engine.is_date_not_required,
        "isDataType": engine.is_data_type,
        "isDataTypeNotRequired": engine.is_data_type_not_required,
        "isMinValue": engine.is_min_value,
        "isMaxValue": engine.is_max_value,
        "IsInAllowedValues": engine.is_in_allowed_values,
        "dataLengthCheck": engine.data_length_check,
        "isMONCCYYValid": engine.is_mon_ccyy_valid,
        "IsInAllowedValuesAndNull": engine.is_in_allowed_values_and_null,
        "isRequiredWithCondition": engine.is_required_with_condition,
        "IsOverPunch": engine.is_over_punch
    }

    output_payload = {}
    try:
        ctx = dbutils.notebook.getContext()
        output_payload["CurrentJobId"] = str(ctx.tags.get("jobId", ""))
        
        # Ingest raw targeted data files
        if ignore_header == "true" and header == "true":
            df_file = is_ignore_header(data_path, schema_path, delimiter, text_qualifier)
        elif ignore_header == "false" and header == "false":
            df_file = without_header(data_path, schema_path, delimiter, text_qualifier)
        else:
            df_file = spark_session.read.option("header", header).format("csv") \
                .option("delimiter", delimiter).option("quote", text_qualifier) \
                .option("inferSchema", "true").load(data_path)

        df_data1 = df_file.withColumn("FILE_ID", F.lit(file_id)) \
            .withColumn("ROW_ID1", F.monotonically_increasing_id() + 1)
        
        window_spec = Window.partitionBy("FILE_ID").orderBy("ROW_ID1")
        df_data = df_data1.withColumn("ROW_ID", F.row_number().over(window_spec)).drop("ROW_ID1")
        
        # Load constraints validation mapping file schema rules
        df_schema = spark_session.read.format("json").option("multiline", "true").load(schema_path)
        exploded_validations = df_schema.select(F.explode("ColumnNames").alias("col")) \
            .select("col.FieldName", "col.Validators").filter(F.col("FieldName") != "TEMPLATE")
            
        df_validators = exploded_validations.withColumn("Validators", F.explode("Validators")) \
            .select(
                F.col("FieldName"),
                F.col("Validators.Name").alias("Name"),
                F.col("Validators.Value").alias("Value"),
                F.col("Validators.InputValue").alias("InputValue"),
                F.col("Validators.ConditionCol").alias("ConditionCol"),
                F.col("Validators.ConditionVal").alias("ConditionVal")
            )

        validation_outcome = 0
        # Driving rules iteratively over dynamic configurations parsed locally safely
        for row in df_validators.collect():
            rule = row.asDict()
            v_name = str(rule.get("Name"))
            
            if v_name in validator_registry:
                validation_outcome += validator_registry[v_name](
                    df_data=df_data,
                    col_name=str(rule.get("FieldName")),
                    validator=v_name,
                    error_path=full_error,
                    **rule
                )

        output_payload.update({
            "Status": "SUCCESS",
            "RecordCount": str(df_data.count()),
            "ErrorCount": str(validation_outcome),
            "ErrorPathSchema": full_error,
            "ErrorMessage": ""
        })

    except Exception as e:
        output_payload.update({
            "Status": "FAILED",
            "RecordCount": "0",
            "ErrorCount": "0",
            "ErrorPathSchema": full_error,
            "ErrorMessage": str(e).replace('"', '')
        })
    finally:
        dbutils.notebook.exit(json.dumps(output_payload))

In [0]:
# ==============================================================================
# Executing Engine Call Lifecycle Pass
# ==============================================================================
run_level2_validator(full_validation, full_file)